> **Inputs:** This notebook expects the original processed development and held-out Parquet files. They are not shipped here. Its numerical outputs belong to the primary analysis, not the separately reconstructed 06C stream.


# Notebook 06B — Preprocessing mechanism and transition validation
**IEEE J-BHI reliability-aware patient segmentation** · development-cohort sensitivity audit

This notebook **does not overwrite Notebook 06**. It reproduces its main mechanism, corrects paired-distance sampling, separates utilization counts from observation indicators, and adds K-means objective decomposition, seed sensitivity, scaling/K sensitivity, and a shuffled-utilization negative control. Outputs are saved to `results/v11_scaling_audit_06B/`.

**Interpretation safeguards:** (i) ARI≈0 vs utilization-only is **not** evidence of clinical validity; (ii) inertia contributions are *within fitted clusters*, not feature importance or causality; (iii) K-means random starts and weights can change solutions; (iv) development-only evidence is not external validation. The original 06 results remain historical evidence.

In [ ]:
# Run once if your Jupyter kernel lacks packages, then restart kernel.
# %pip install numpy pandas scikit-learn scipy matplotlib pyarrow joblib
from pathlib import Path
import sys, json, hashlib, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from IPython.display import display

PROJECT = Path(r'C:\Projects\healthcare-cluster-reliability')
DATA = PROJECT / 'data' / 'processed'
OUT = PROJECT / 'results' / 'v11_scaling_audit_06B'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], 'scikit-learn:', sklearn.__version__, 'pandas:', pd.__version__)
print('Data folder:', DATA, 'exists:', DATA.exists())
assert DATA.exists(), 'Change PROJECT to your local project directory.'

# Default run: ~200-300 K-means fits; may take several minutes on Windows.
SEEDS = [42, 101, 202, 303, 404]
N_INIT = 10
SIL_SAMPLE = 1500
WEIGHTS = [0.0, 0.10, 0.25, 0.50, 0.55, 0.60, 0.625, 0.65, 0.675, 0.70, 0.725, 0.75, 1.0, 1.5, 2.0]
K_VALUES = [2, 3, 5, 8]
CAT = ['age','diag1_family','diag2_family','diag3_family']
UTIL = ['number_outpatient','number_emergency','number_inpatient']
OBS = ['A1Cresult_observed','max_glu_serum_observed']


## 1. Load and validate development feature tables

In [ ]:
dev = {}
for rep in ['R1','R2','R3']:
    path = DATA / f'development_{rep}_10k.parquet'
    assert path.exists(), f'Missing {path}; run earlier cohort/feature notebooks.'
    dev[rep] = pd.read_parquet(path).reset_index(drop=True)
    required = CAT + (UTIL if rep != 'R1' else []) + (OBS if rep == 'R3' else []) + ['patient_nbr']
    assert set(required).issubset(dev[rep].columns), f'{rep} missing {set(required)-set(dev[rep].columns)}'
    assert dev[rep]['patient_nbr'].is_unique, f'{rep}: repeated patient IDs'
    print(rep, dev[rep].shape)
assert all(dev['R1']['patient_nbr'].equals(dev[r]['patient_nbr']) for r in ['R2','R3']), 'Patient ordering differs'
assert len(dev['R2']) == 10000, 'Unexpected development cohort size'


## 2. Historical preprocessing (most-frequent categorical imputation) and fitted-scale audit

In [ ]:
def make_pre(rep='R2', scaler='robust', categorical_strategy='most_frequent'):
    cat_imp = (SimpleImputer(strategy='most_frequent') if categorical_strategy=='most_frequent'
               else SimpleImputer(strategy='constant', fill_value='missing'))
    scale = {'robust': RobustScaler, 'standard': StandardScaler}.get(scaler)
    num_scaler = scale() if scale else 'passthrough'
    num_cols = UTIL + (OBS if rep == 'R3' else [])
    blocks = [('cat', Pipeline([('imp',cat_imp),('oh',OneHotEncoder(handle_unknown='ignore'))]), CAT)]
    if num_cols:
        blocks.append(('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',num_scaler)]),num_cols))
    return ColumnTransformer(blocks, sparse_threshold=0.0)

def dense(x):
    return x.toarray() if hasattr(x,'toarray') else np.asarray(x)

def prep_blocks(rep='R2', scaler='robust'):
    p=make_pre(rep,scaler).fit(dev[rep])
    x=dense(p.transform(dev[rep])).astype(np.float64,copy=False)
    cat=x[:,p.output_indices_['cat']]
    num=x[:,p.output_indices_['num']]
    u=num[:,:3]
    obs=num[:,3:] if rep=='R3' else np.empty((len(x),0))
    return p,cat,u,obs

prepared = {(rep,scale):prep_blocks(rep,scale) for rep in ['R2','R3'] for scale in ['robust','standard','none']}
p,cat,u,obs=prepared[('R2','robust')]
sc=p.named_transformers_['num'].named_steps['scale']
scale_rows=[]
for j,name in enumerate(UTIL):
    s=pd.to_numeric(dev['R2'][name],errors='coerce')
    scale_rows.append({'feature':name,'zero_fraction':float((s==0).mean()),'median':float(s.median()),
                       'IQR':float(s.quantile(.75)-s.quantile(.25)),
                       'fitted_center':float(sc.center_[j]),'fitted_scale':float(sc.scale_[j])})
scale_audit=pd.DataFrame(scale_rows)
display(scale_audit)
scale_audit.to_csv(OUT/'01_fitted_scaler_audit.csv',index=False)


## 3. Corrected pairwise distance sampling — aligned pairs, no independent filtering

In [ ]:
rng=np.random.default_rng(42)
n=len(cat); a=rng.integers(0,n,size=50000); bb=rng.integers(0,n,size=50000)
keep=(a!=bb); a=a[keep]; bb=bb[keep]
assert len(a)==len(bb) and np.all(a!=bb)
blocks={'clinical_onehot':cat,'utilization_counts':u}
rows=[]
for name,arr in blocks.items():
    d=np.sum((arr[a]-arr[bb])**2,axis=1)
    rows.append({'block':name,'dimensions':arr.shape[1], 'pairs':len(d),
                 'mean_pair_sq_distance':float(d.mean()),'median_pair_sq_distance':float(np.median(d))})
distance_df=pd.DataFrame(rows)
distance_df['share_of_mean_distance']=distance_df.mean_pair_sq_distance/distance_df.mean_pair_sq_distance.sum()
display(distance_df)
distance_df.to_csv(OUT/'02_corrected_pairwise_distances.csv',index=False)
fig,ax=plt.subplots(figsize=(6.4,4)); ax.bar(distance_df.block,distance_df.mean_pair_sq_distance)
ax.set(ylabel='Mean pairwise squared distance',title='R2 feature-block geometry (corrected paired sampling)')
ax.tick_params(axis='x',rotation=12); fig.tight_layout()
fig.savefig(OUT/'fig_01_pairwise_block_geometry.png',dpi=250);fig.savefig(OUT/'fig_01_pairwise_block_geometry.pdf');plt.show()


## 4. Objective decomposition and seed sensitivity — utilization-only reweighting
**Crucial change from Notebook 06:** R3 observation flags are held fixed at weight 1.0 while only the three utilization counts are reweighted. `w=0` removes utilization counts, **not** R3 observation indicators.

In [ ]:
def km(x,k=2,seed=42):
    model=KMeans(n_clusters=k,n_init=N_INIT,random_state=seed)
    labels=model.fit_predict(x)
    return model,labels

def assemble(cat,u,obs,w):
    return np.hstack([cat,w*u,obs])

def inertia_blocks(x,model,labels,groups):
    mu=model.cluster_centers_[labels]
    total=np.sum((x-mu)**2)
    global_mean=x.mean(axis=0)
    between_total=0.0
    results={}
    for name,sl in groups.items():
        within=float(np.sum((x[:,sl]-mu[:,sl])**2))
        between=float(np.sum((mu[:,sl]-global_mean[sl])**2))
        results[f'{name}_within']=within
        results[f'{name}_between']=between
        between_total+=between
    for name in groups:
        results[f'{name}_within_share']=results[f'{name}_within']/total if total else np.nan
        results[f'{name}_between_share']=results[f'{name}_between']/between_total if between_total else np.nan
    return results

rows=[]
for rep in ['R2','R3']:
    p,c,u,o=prepared[(rep,'robust')]
    util_model,util_labels=km(u,2,42)
    base_model,base_labels=km(assemble(c,u,o,1.0),2,42)
    groups={'clinical':slice(0,c.shape[1]),'utilization':slice(c.shape[1],c.shape[1]+3)}
    if rep=='R3':groups['observation']=slice(c.shape[1]+3,c.shape[1]+3+o.shape[1])
    for w in WEIGHTS:
        x=assemble(c,u,o,w)
        for seed in SEEDS:
            model,labels=km(x,2,seed)
            counts=np.bincount(labels,minlength=2)
            record={'representation':rep,'weight':w,'seed':seed,'k':2,
                    'ARI_vs_utilonly':adjusted_rand_score(labels,util_labels),
                    'ARI_vs_weight1':adjusted_rand_score(labels,base_labels),
                    'smallest_cluster':int(counts.min()),'largest_cluster':int(counts.max()),
                    'silhouette_sample':float(silhouette_score(x,labels,sample_size=min(SIL_SAMPLE,len(x)),random_state=42)),
                    'total_inertia':float(model.inertia_)}
            record.update(inertia_blocks(x,model,labels,groups))
            rows.append(record)
weight_runs=pd.DataFrame(rows)
weight_runs.to_csv(OUT/'03_weight_seed_objective_detail.csv',index=False)
metrics=['ARI_vs_utilonly','ARI_vs_weight1','smallest_cluster','silhouette_sample','clinical_within_share','utilization_within_share','clinical_between_share','utilization_between_share']
weight_summary=weight_runs.groupby(['representation','weight'])[metrics].agg(['mean','std','min','max'])
weight_summary.columns=['_'.join(col) for col in weight_summary.columns]
weight_summary=weight_summary.reset_index()
display(weight_summary[['representation','weight','ARI_vs_utilonly_mean','ARI_vs_utilonly_min','ARI_vs_utilonly_max','smallest_cluster_mean','silhouette_sample_mean']])
weight_summary.to_csv(OUT/'04_weight_seed_summary.csv',index=False)


## 5. Publication figures — weight transition, silhouette, cluster size, inertia

In [ ]:
def plot_summary(metric,ylabel,filename,title,ylim=None):
    fig,ax=plt.subplots(figsize=(7.4,4.6))
    for rep,g in weight_summary.groupby('representation'):
        g=g.sort_values('weight');xx=g.weight.to_numpy()
        yy=g[f'{metric}_mean'].to_numpy();lo=g[f'{metric}_min'].to_numpy();hi=g[f'{metric}_max'].to_numpy()
        line=ax.plot(xx,yy,'o-',label=rep,linewidth=1.8,markersize=4)[0]
        ax.fill_between(xx,lo,hi,color=line.get_color(),alpha=.14)
    ax.set(xlabel='Utilization-count weight (observation flags held fixed)',ylabel=ylabel,title=title)
    if ylim:ax.set_ylim(*ylim)
    ax.legend(title='Representation');ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(OUT/(filename+'.png'),dpi=300);fig.savefig(OUT/(filename+'.pdf'));plt.show()

plot_summary('ARI_vs_utilonly','ARI versus utilization-only','fig_02_transition_ARI','Utilization-only agreement across seeds',(-.1,1.05))
plot_summary('silhouette_sample','Silhouette (fixed sample size)','fig_03_transition_silhouette','Conventional quality across feature weights')
plot_summary('smallest_cluster','Smallest cluster size (patients)','fig_04_transition_cluster_size','Cluster composition across feature weights')
plot_summary('utilization_between_share','Utilization share of between-cluster separation','fig_05_between_cluster_share','Feature-block separation mechanism',(0,1))
plot_summary('utilization_within_share','Utilization share of within-cluster inertia','fig_06_within_cluster_share','Feature-block inertia decomposition',(0,1))


## 6. Seed agreement at each weight (pairwise ARI among independently initialized fits)

In [ ]:
seed_rows=[]
for rep in ['R2','R3']:
    _,c,u,o=prepared[(rep,'robust')]
    for w in WEIGHTS:
        x=assemble(c,u,o,w)
        label_sets=[km(x,2,seed)[1] for seed in SEEDS]
        aris=[adjusted_rand_score(label_sets[i],label_sets[j]) for i in range(len(SEEDS)) for j in range(i+1,len(SEEDS))]
        seed_rows.append({'representation':rep,'weight':w,'seed_pair_ARI_mean':np.mean(aris),'seed_pair_ARI_min':np.min(aris),'seed_pair_ARI_std':np.std(aris)})
seed_stability=pd.DataFrame(seed_rows)
display(seed_stability)
seed_stability.to_csv(OUT/'05_seed_pairwise_stability.csv',index=False)
fig,ax=plt.subplots(figsize=(7.4,4.4))
for rep,g in seed_stability.groupby('representation'):
    ax.plot(g.weight,g.seed_pair_ARI_mean,'o-',label=rep)
ax.set(xlabel='Utilization-count weight',ylabel='Mean pairwise seed ARI',title='Initialization reproducibility by weight',ylim=(-.05,1.05));ax.legend();ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'fig_07_seed_stability.png',dpi=300);fig.savefig(OUT/'fig_07_seed_stability.pdf');plt.show()


## 7. Scaler × K comparison (same preprocessing and independent utilization-only fit)

In [ ]:
scale_rows=[]
for rep in ['R2','R3']:
    for scale in ['robust','standard','none']:
        p,c,u,o=prepared[(rep,scale)]
        x=assemble(c,u,o,1.0)
        for k in K_VALUES:
            _,lab=km(x,k,42)
            _,lab_u=km(u,k,42)
            scale_rows.append({'representation':rep,'scaler':scale,'k':k,
                               'ARI_full_vs_utilonly':adjusted_rand_score(lab,lab_u),
                               'full_min_cluster':int(np.bincount(lab,minlength=k).min()),
                               'util_min_cluster':int(np.bincount(lab_u,minlength=k).min())})
scale_k=pd.DataFrame(scale_rows)
display(scale_k)
scale_k.to_csv(OUT/'06_scaler_K_comparison.csv',index=False)
for rep in ['R2','R3']:
    pivot=scale_k[scale_k.representation==rep].pivot(index='scaler',columns='k',values='ARI_full_vs_utilonly').loc[['robust','standard','none'],K_VALUES]
    fig,ax=plt.subplots(figsize=(6.6,3.4));im=ax.imshow(pivot.values,vmin=-.1,vmax=1,cmap='viridis',aspect='auto')
    ax.set_xticks(range(len(pivot.columns)),pivot.columns);ax.set_yticks(range(len(pivot.index)),pivot.index)
    ax.set(xlabel='K',ylabel='Numerical scaler',title=f'{rep}: full vs utilization-only ARI')
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val=pivot.iloc[i,j];ax.text(j,i,f'{val:.3f}',ha='center',va='center',color='white' if val<.5 else 'black',fontsize=10)
    fig.colorbar(im,ax=ax,label='ARI');fig.tight_layout()
    fig.savefig(OUT/f'fig_08_scaler_K_heatmap_{rep}.png',dpi=300);fig.savefig(OUT/f'fig_08_scaler_K_heatmap_{rep}.pdf');plt.show()


## 8. Negative control: independently permute utilization columns across patients
Preserves each count column’s empirical marginal distribution, but destroys its patient-level association with clinical/observation variables and cross-utilization relationships. **Not** a causal test or clinical validation.

In [ ]:
control_rows=[]
CONTROL_SEEDS=[42,101,202]
CONTROL_WEIGHTS=[0.5,0.6,0.65,0.7,0.75,1.0]
for rep in ['R2','R3']:
    _,c,u,o=prepared[(rep,'robust')]
    for shuffle_seed in CONTROL_SEEDS:
        rng=np.random.default_rng(shuffle_seed)
        shuffled=np.column_stack([rng.permutation(u[:,j]) for j in range(u.shape[1])])
        for kind,uu in [('observed',u),('shuffled',shuffled)]:
            _,util_lab=km(uu,2,42)
            for w in CONTROL_WEIGHTS:
                x=assemble(c,uu,o,w)
                _,labels=km(x,2,42)
                control_rows.append({'representation':rep,'shuffle_seed':shuffle_seed,'condition':kind,
                                     'weight':w,'ARI_full_vs_utilonly':adjusted_rand_score(labels,util_lab),
                                     'min_cluster':int(np.bincount(labels,minlength=2).min())})
controls=pd.DataFrame(control_rows)
controls.to_csv(OUT/'07_negative_control.csv',index=False)
control_summary=controls.groupby(['representation','condition','weight'],as_index=False)['ARI_full_vs_utilonly'].agg(['mean','std']).reset_index()
display(control_summary)
for rep in ['R2','R3']:
    fig,ax=plt.subplots(figsize=(7,4.3))
    for kind,g in control_summary[control_summary.representation==rep].groupby('condition'):
        ax.plot(g.weight,g['mean'],'o-',label=kind)
    ax.set(xlabel='Utilization-count weight',ylabel='ARI vs condition-specific utilization-only partition',
           title=f'{rep}: observed vs shuffled-utilization negative control',ylim=(-.1,1.05))
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(OUT/f'fig_09_negative_control_{rep}.png',dpi=300);fig.savefig(OUT/f'fig_09_negative_control_{rep}.pdf');plt.show()


## 9. Optional held-out evaluation — only if matching feature files exist
This section does **not** refit on held-out data. Preprocessors and cluster centers are fitted on development data. The utilization-only reference model is also fitted on development and then applied to held-out data. Check that the feature schema and patient identifiers match.

In [ ]:
held={}
for rep in ['R2','R3']:
    path=DATA/f'heldout_{rep}_61k.parquet'
    if path.exists():
        held[rep]=pd.read_parquet(path).reset_index(drop=True)
if not held:
    print('Held-out feature tables absent: development-only audit. No held-out claims.')
else:
    hold_rows=[]
    for rep,df in held.items():
        assert set(df.patient_nbr).isdisjoint(set(dev[rep].patient_nbr)), 'PATIENT LEAKAGE'
        for scale in ['robust','standard']:
            pre=make_pre(rep,scale).fit(dev[rep])
            xdev=dense(pre.transform(dev[rep]));xhold=dense(pre.transform(df))
            cat_sl=pre.output_indices_['cat'];num_sl=pre.output_indices_['num']
            cdev=xdev[:,cat_sl];c_hold=xhold[:,cat_sl]
            udev=xdev[:,num_sl][:,:3];u_hold=xhold[:,num_sl][:,:3]
            odev=xdev[:,num_sl][:,3:] if rep=='R3' else np.empty((len(xdev),0))
            o_hold=xhold[:,num_sl][:,3:] if rep=='R3' else np.empty((len(xhold),0))
            util_model,_=km(udev,2,42)
            hold_util_labels=util_model.predict(u_hold)
            for w in [0.5,0.6,0.65,0.7,0.75,1.0]:
                full_model,_=km(assemble(cdev,udev,odev,w),2,42)
                hold_full_labels=full_model.predict(assemble(c_hold,u_hold,o_hold,w))
                hold_rows.append({'representation':rep,'scaler':scale,'weight':w,'n_heldout':len(df),
                                  'heldout_ARI_full_vs_utilonly':adjusted_rand_score(hold_full_labels,hold_util_labels),
                                  'heldout_min_cluster':int(np.bincount(hold_full_labels,minlength=2).min())})
    held_results=pd.DataFrame(hold_rows)
    display(held_results)
    held_results.to_csv(OUT/'08_heldout_fixed_model_optional.csv',index=False)


## 10. Manifest and interpretation checklist
**Before publication:** check actual figure labels and readability; report how weights act on *squared* distances (weighting a feature by `w` weights its squared-distance term by `w²`); check sensitivity to K, seeds, and subgroup composition; distinguish development evidence from held-out; do not claim clinical phenotype validity from ARI alone.

In [ ]:
manifest={'source_notebook':'06B_preprocessing_mechanism_validation.ipynb',
          'project':str(PROJECT),'cohort':'development 10,000 patients','data_files':{},
          'seeds':SEEDS,'n_init':N_INIT,'weights':WEIGHTS,'k_values':K_VALUES,
          'categorical_imputation':'most_frequent (historical Notebook 06 configuration)',
          'numerical_imputation':'median','R3_observation_flags_weight':1.0,
          'python':sys.version,'pandas':pd.__version__,'sklearn':sklearn.__version__,
          'note':'Development-only unless 08_heldout_fixed_model_optional.csv exists.'}
for rep in ['R1','R2','R3']:
    path=DATA/f'development_{rep}_10k.parquet'
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    manifest['data_files'][rep]={'path':str(path),'sha256':h.hexdigest(),'rows':len(dev[rep])}
(OUT/'09_run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('All outputs saved to:',OUT)
print('Files:',*[p.name for p in sorted(OUT.iterdir())],sep='\n  ')
